# 01 · MF-DFA extraction

Computes the MF-DFA multifractal descriptors over the 14 segments of every
painting, in two scale bands, and stores them in `mfdfa_b1` and `mfdfa_b2`.

| Band | Scale range | Integration | Trend order | Ladder density |
|---|---|---|---|---|
| `b1` | 6 px to 25% of the side | yes | 2 | √2 |
| `b2` | 25% to 75% of the side | yes | 2 | 2^(1/4) |

**The run is resumable.** A row in the table means that painting is done, so if
the run is interrupted just execute the extraction cell again: it picks up where
it stopped. Failures are recorded in `extraction_log` with their message and do
not stop the sweep.

Rough cost: a few seconds per painting in sequential mode.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd

from datasets import load_dataset

import utils as ut
import mfdfa
import db

## Parameters

Both bands share the integration flag and the trend order; what separates them
is the scale range and the density of the logarithmic ladder.

In [ ]:
MAX_SIZE = 1380          # side of the normalised canvas
COMMIT_EVERY = 50        # paintings per transaction
REPORT_EVERY = 100       # paintings per progress line
LIMIT = None             # None processes everything; a number caps the run

BAND_B1 = dict(q_min=-5.0, q_max=5.0, s_min=6,    s_max=0.25,
               integration=True, degree_trend=2, degree_scales=1)

BAND_B2 = dict(q_min=-5.0, q_max=5.0, s_min=0.25, s_max=0.75,
               integration=True, degree_trend=2, degree_scales=2)

## Current state

`create_extraction_log` is idempotent: if the table already exists it does
nothing.

In [ ]:
con = db.connect()
db.create_extraction_log(con)

total = con.execute("SELECT COUNT(*) FROM metadata").fetchone()[0]
done = db.done_painting_ids(con, "mfdfa_b1") & db.done_painting_ids(con, "mfdfa_b2")
failed = db.failed_painting_ids(con, "mfdfa", "b1b2")

print(f"paintings in metadata : {total}")
print(f"already processed     : {len(done)}")
print(f"failed before         : {len(failed)}")
print(f"pending               : {total - len(done)}")

## Per-painting extraction

Returns the two feature dictionaries with the column names already assembled.
If one segment fails, its features are left missing instead of discarding the
whole painting.

In [ ]:
def extract_one(img_norm):
    """MF-DFA over the 14 segments, in both bands."""
    feats_b1, feats_b2 = {}, {}
    failures = []

    for grid in range(1, 4):
        for pos, seg in enumerate(ut.segment_image(img_norm, grid_size=grid)):
            name_seg = f"seg{grid}{pos + 1}"

            try:
                _, f1 = mfdfa.mf_dfa_features(seg, **BAND_B1)
                feats_b1.update({f"{k}/{name_seg}/b1": v for k, v in f1.items()})
            except Exception as e:
                failures.append(f"b1/{name_seg}: {e}")

            try:
                _, f2 = mfdfa.mf_dfa_features(seg, **BAND_B2)
                feats_b2.update({f"{k}/{name_seg}/b2": v for k, v in f2.items()})
            except Exception as e:
                failures.append(f"b2/{name_seg}: {e}")

    return feats_b1, feats_b2, failures

## Single-painting smoke test

Before launching the long run: check that the column names match the schema,
how many features come out, and how many are NaN. A name mismatch shows up
here, not twelve hours later.

In [ ]:
ds_test = load_dataset("huggan/wikiart", split="train",
                      streaming=True, columns=["image"])
item = next(iter(ds_test))

img_norm = ut.normalize_image(img=np.array(item["image"]),
                              max_size=MAX_SIZE, gray=True)
t0 = time.time()
f1, f2, failures = extract_one(img_norm)
dt = time.time() - t0

expected = set(db.feature_columns(db.FEATURES_DFA, "b1"))
obtained = set(f1)

print(f"seconds per painting : {dt:.2f}")
print(f"features             : {len(f1)} in b1, {len(f2)} in b2")
print(f"NaN in b1            : {sum(np.isnan(v) for v in f1.values())}")
print(f"NaN in b2            : {sum(np.isnan(v) for v in f2.values())}")
print(f"missing columns      : {sorted(expected - obtained)[:5]}")
print(f"unexpected columns   : {sorted(obtained - expected)[:5]}")
print(f"failed segments      : {failures[:3]}")

## Full extraction

This is the long cell. It can be interrupted with the stop button without
losing committed work: the `finally` block closes the pending transaction.

To resume, run the state cell above and then this one again.

In [ ]:
dataset = load_dataset("huggan/wikiart", split="train",
                      streaming=True, columns=["image"])

n_ok = n_err = n_skip = 0
t0 = time.time()

try:
    for painting_id, item in enumerate(dataset):
        if painting_id in done:
            n_skip += 1
            continue
        if LIMIT is not None and n_ok + n_err >= LIMIT:
            break

        try:
            img_norm = ut.normalize_image(img=np.array(item["image"]),
                                          max_size=MAX_SIZE, gray=True)
            feats_b1, feats_b2, failures = extract_one(img_norm)

            db.insert_features(con, "mfdfa_b1", painting_id, feats_b1, commit=False)
            db.insert_features(con, "mfdfa_b2", painting_id, feats_b2, commit=False)
            db.log_extraction(con, painting_id, "mfdfa", "b1b2",
                              "ok" if not failures else "error",
                              error=None if not failures else "; ".join(failures),
                              commit=False)
            n_ok += 1

        except Exception as e:
            db.log_extraction(con, painting_id, "mfdfa", "b1b2",
                              "error", error=e, commit=False)
            n_err += 1

        if (n_ok + n_err) % COMMIT_EVERY == 0:
            con.commit()

        if (n_ok + n_err) % REPORT_EVERY == 0:
            elapsed = time.time() - t0
            rate = (n_ok + n_err) / elapsed
            left = total - len(done) - (n_ok + n_err)
            print(f"{n_ok + n_err:>6} processed  "
                  f"{n_err:>4} failed  "
                  f"{rate:.2f} paintings/s  "
                  f"~{left / rate / 3600:.1f} h to go")

finally:
    con.commit()
    print(f"\ncommitted: {n_ok} ok, {n_err} failed, "
          f"{n_skip} skipped as already done")

## Verification

How many rows landed, how many failures remain, and where the missing values
concentrate.

In [ ]:
pd.read_sql_query("""
    SELECT status, COUNT(*) AS paintings
    FROM extraction_log
    WHERE method = 'mfdfa'
    GROUP BY status
""", con)

In [ ]:
pd.read_sql_query("""
    SELECT painting_id, substr(error, 1, 120) AS error
    FROM extraction_log
    WHERE method = 'mfdfa' AND status = 'error'
    LIMIT 10
""", con)

### Missing values per feature

A high count on one column points at a systematic problem, for example a
regression that fails to converge on the small segments of band 2.

In [ ]:
cols = db.feature_columns(db.FEATURES_DFA, "b2")
parts = ", ".join(f'COUNT(*) - COUNT("{c}") AS "{c}"' for c in cols[:14])

nulls = pd.read_sql_query(f"SELECT {parts} FROM mfdfa_b2", con).T
nulls.columns = ["nulls"]
nulls.sort_values("nulls", ascending=False).head(14)

In [ ]:
con.close()